# ExDark – Exploratory Data Analysis
Covers class balance, split sanity, image properties, annotation statistics, illumination and data-quality checks.
Run from the `notebooks/` directory (paths are relative to the project root `..`).

In [ ]:
from ultralytics import YOLO
import pandas as pd
import torch
from pathlib import Path

In [ ]:
import hashlib
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3})

ROOT_PATH = Path("..").resolve()
EXDARK_PATH = "data/raw/images/ExDark"
EXDARK_DIR = ROOT_PATH / EXDARK_PATH
ANNO_DIR = ROOT_PATH / "data/raw/annotations/ExDark_Annno"
OUT_DIR = ROOT_PATH / "data/outputs"
CACHE = OUT_DIR / "eda_image_stats.csv"   # cached per-image stats (delete to recompute)
CLASSES = sorted(p.name for p in EXDARK_DIR.iterdir() if p.is_dir())
print(len(CLASSES), "classes:", CLASSES)

## 1. Images per class and train/val/test split

In [ ]:
df = pd.read_csv(OUT_DIR / "classes.csv")
splits = {s: pd.read_csv(OUT_DIR / f"splits/{s}.csv") for s in ["train", "validation", "test"]}
df["split"] = df["image"].map({img: s for s, d in splits.items() for img in d["image"]})
print(f"{len(df)} images; split sizes:", df["split"].value_counts().to_dict())

counts = df["class"].value_counts().sort_values()
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(counts.index, counts.values, color="#4C78A8")
for i, v in enumerate(counts.values):
    ax.text(v + 10, i, str(v), va="center")
ax.set(title="Images per class (class = folder)", xlabel="images")
plt.show()
print(f"imbalance ratio (max/min): {counts.max() / counts.min():.2f}")

# stratification check: share of each class inside each split
share = pd.crosstab(df["class"], df["split"], normalize="columns").mul(100).round(2)
share["max_gap_pp"] = (share.max(axis=1) - share.min(axis=1)).round(2)
share

## 2. Image properties (size, format, file size)

In [ ]:
def load_stats():
    if CACHE.exists():
        return pd.read_csv(CACHE)
    return None

stats = load_stats()
if stats is None or len(stats) != len(df):
    rows = []
    for img, cls in zip(df["image"], df["class"]):
        p = ROOT_PATH / img
        row = {"image": img, "class": cls, "ext": p.suffix.lower(), "bytes": p.stat().st_size,
               "ok": True, "w": np.nan, "h": np.nan, "mode": None,
               "brightness": np.nan, "dark_frac": np.nan, "md5": None}
        try:
            with Image.open(p) as im:
                row["w"], row["h"], row["mode"] = im.width, im.height, im.mode
                im.draft("L", (128, 128))                      # fast JPEG downscale on decode
                g = np.asarray(im.convert("L").resize((96, 96)), dtype=np.float32) / 255
            row["brightness"] = g.mean()                       # mean luminance, 0..1
            row["dark_frac"] = (g < 0.1).mean()                # share of near-black pixels
        except Exception as e:                                 # unreadable / truncated file
            row["ok"] = False
            print("unreadable:", img, type(e).__name__)
        row["md5"] = hashlib.md5(p.read_bytes()).hexdigest()
        rows.append(row)
    stats = pd.DataFrame(rows)
    stats.to_csv(CACHE, index=False)

stats = stats.merge(df[["image", "split"]], on="image")
stats["megapixels"] = stats.w * stats.h / 1e6
stats["aspect"] = stats.w / stats.h
print("unreadable images:", (~stats.ok).sum())
print(stats[["w", "h", "megapixels", "aspect", "bytes"]].describe().round(2))
print("\nformats:", stats.ext.value_counts().to_dict(), "| modes:", stats["mode"].value_counts().to_dict())

In [ ]:
top = stats.groupby(["w", "h"]).size().sort_values(ascending=False).head(10)
print("most common resolutions:\n", top)
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
axes[0].hist(stats.megapixels.dropna(), bins=50, color="#4C78A8"); axes[0].set(title="Megapixels", xlabel="MP")
axes[1].hist(stats.aspect.dropna().clip(0.3, 3), bins=50, color="#F58518"); axes[1].set(title="Aspect ratio (w/h)")
axes[2].hist(stats.bytes / 1e6, bins=50, color="#54A24B"); axes[2].set(title="File size", xlabel="MB")
plt.tight_layout(); plt.show()

## 3. Annotations (bounding boxes)
Files are in Piotr-Dollar bbGt format: `label l t w h ...`, pixel coordinates.

In [ ]:
def find_anno(cls, image_name):
    exact = ANNO_DIR / cls / f"{image_name}.txt"
    if exact.exists():
        return exact
    stem = Path(image_name).stem   # tolerate extension-case mismatches (.JPG vs .jpg)
    return next((a for e in (".jpg", ".JPG", ".jpeg", ".JPEG", ".png", ".PNG")
                 if (a := ANNO_DIR / cls / f"{stem}{e}.txt").exists()), None)

boxes = []
for img, cls in zip(df["image"], df["class"]):
    f = find_anno(cls, Path(img).name)
    if f is None:
        print("missing annotation:", img); continue
    for line in f.read_text().splitlines():
        parts = line.split()
        if parts and not line.startswith("%"):
            boxes.append((img, cls, parts[0], *map(float, parts[1:5])))
boxes = pd.DataFrame(boxes, columns=["image", "folder", "label", "l", "t", "w", "h"])
boxes = boxes.merge(stats[["image", "w", "h", "split", "brightness"]].rename(columns={"w": "W", "h": "H"}), on="image")

# normalised geometry, and area relative to a 640-px-long-side resize (what YOLO sees)
boxes["rel_w"], boxes["rel_h"] = boxes.w / boxes.W, boxes.h / boxes.H
boxes["rel_area"] = boxes.rel_w * boxes.rel_h
scale = 640 / boxes[["W", "H"]].max(axis=1)
boxes["area640"] = boxes.w * boxes.h * scale**2
boxes["cx"], boxes["cy"] = (boxes.l + boxes.w / 2) / boxes.W, (boxes.t + boxes.h / 2) / boxes.H
boxes["ar"] = boxes.w / boxes.h.replace(0, np.nan)
print(f"{len(boxes)} boxes in {boxes.image.nunique()} images")

In [ ]:
per_img = boxes.groupby("image").size().reindex(df["image"], fill_value=0)
print("images with no boxes:", (per_img == 0).sum())
print(per_img.describe().round(2))
print("\nboxes whose label != image folder:", f"{(boxes.label != boxes.folder).mean():.1%}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
bc = boxes.label.value_counts().sort_values()
axes[0].barh(bc.index, bc.values, color="#4C78A8")
for i, v in enumerate(bc.values): axes[0].text(v + 20, i, str(v), va="center")
axes[0].set(title="Boxes per class", xlabel="boxes")
axes[1].hist(per_img.clip(upper=20), bins=np.arange(0, 22) - 0.5, color="#F58518")
axes[1].set(title="Objects per image (clipped at 20)", xlabel="boxes in image")
plt.tight_layout(); plt.show()

In [ ]:
# Box size: COCO buckets measured at 640px input (small < 32², medium < 96², large)
bucket = pd.cut(boxes.area640, [0, 32**2, 96**2, np.inf], labels=["small", "medium", "large"])
size_tbl = pd.crosstab(boxes.label, bucket, normalize="index").mul(100).round(1)
print("overall:", bucket.value_counts(normalize=True).mul(100).round(1).to_dict())
display(size_tbl) if "display" in globals() else print(size_tbl)

order = boxes.groupby("label").rel_area.median().sort_values().index
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
axes[0].boxplot([np.log10(boxes.rel_area[boxes.label == c].clip(lower=1e-6)) for c in order],
                vert=False, tick_labels=order, showfliers=False)
axes[0].set(title="Box area / image area (log10)", xlabel="log10(rel. area)")
axes[1].hist(boxes.ar.dropna().clip(0.1, 6), bins=60, color="#54A24B"); axes[1].set(title="Box aspect ratio (w/h)")
hm = axes[2].hist2d(boxes.cx.clip(0, 1), boxes.cy.clip(0, 1), bins=40, cmap="magma")
axes[2].invert_yaxis(); axes[2].grid(False); axes[2].set(title="Box centre density", xlabel="x", ylabel="y")
plt.tight_layout(); plt.show()

## 4. Illumination
Mean luminance (0 = black, 1 = white) per image, and the fraction of near-black pixels (< 0.1).

In [ ]:
ok = stats[stats.ok]
print(ok.brightness.describe().round(3))
print("images with mean luminance < 0.15:", f"{(ok.brightness < 0.15).mean():.1%}",
      "| < 0.25:", f"{(ok.brightness < 0.25).mean():.1%}")

fig, axes = plt.subplots(1, 3, figsize=(16, 4.3))
axes[0].hist(ok.brightness, bins=50, color="#4C78A8"); axes[0].set(title="Mean luminance per image", xlabel="luminance")
axes[1].hist(ok.dark_frac, bins=50, color="#E45756"); axes[1].set(title="Fraction of near-black pixels", xlabel="fraction")
order = ok.groupby("class").brightness.median().sort_values().index
axes[2].boxplot([ok.brightness[ok["class"] == c] for c in order], vert=False, tick_labels=order, showfliers=False)
axes[2].set(title="Luminance by class", xlabel="luminance")
plt.tight_layout(); plt.show()

print("mean luminance by split:\n", ok.groupby("split").brightness.agg(["mean", "median", "std"]).round(3))

In [ ]:
# Is detection difficulty tied to darkness? Box size vs. image brightness
q = pd.qcut(boxes.brightness, 4, labels=["darkest 25%", "dark", "bright", "brightest 25%"])
print(boxes.groupby(q, observed=True).agg(boxes=("label", "size"), median_rel_area=("rel_area", "median"),
                                            objs_per_img=("image", lambda s: len(s) / s.nunique())).round(4))

## 5. Sample images with ground truth

In [ ]:
rng = np.random.default_rng(0)

def draw(img_path, bx):
    im = Image.open(ROOT_PATH / img_path).convert("RGB")
    d = ImageDraw.Draw(im)
    lw = max(2, im.width // 250)
    for r in bx.itertuples():
        d.rectangle([r.l, r.t, r.l + r.w, r.t + r.h], outline=(255, 215, 0), width=lw)
        d.text((r.l + 3, r.t + 3), r.label, fill=(255, 215, 0))
    return im

def grid(paths, titles, ncols=4, size=3.4):
    nrows = int(np.ceil(len(paths) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * size, nrows * size * 0.8))
    for ax in np.ravel(axes): ax.axis("off")
    for ax, p, t in zip(np.ravel(axes), paths, titles):
        ax.imshow(draw(p, boxes[boxes.image == p])); ax.set_title(t, fontsize=9)
    plt.tight_layout(); plt.show()

picks = [rng.choice(df.image[df["class"] == c]) for c in CLASSES]
grid(picks, CLASSES)

In [ ]:
# Extremes of the brightness range
srt = ok.sort_values("brightness")
grid(list(srt.image[:8]) + list(srt.image[-4:]),
     [f"darkest {b:.3f}" for b in srt.brightness[:8]] + [f"brightest {b:.2f}" for b in srt.brightness[-4:]])

## 6. Data-quality checks

In [ ]:
issues = {}
issues["unreadable images"] = int((~stats.ok).sum())
issues["images without annotation file"] = int(sum(find_anno(c, Path(i).name) is None for i, c in zip(df.image, df["class"])))
issues["images with 0 boxes"] = int((per_img == 0).sum())
issues["boxes with w<=0 or h<=0"] = int(((boxes.w <= 0) | (boxes.h <= 0)).sum())
issues["boxes partly outside image"] = int(((boxes.l < 0) | (boxes.t < 0) | (boxes.l + boxes.w > boxes.W) | (boxes.t + boxes.h > boxes.H)).sum())
issues["boxes tiny (<8 px side at 640)"] = int((np.sqrt(boxes.area640) < 8).sum())
issues["labels outside the 12 classes"] = int((~boxes.label.isin(CLASSES)).sum())
issues["class-folder duplicates (same name in 2 folders)"] = int(df.image.map(lambda p: Path(p).name).duplicated().sum())

dup = stats[stats.duplicated("md5", keep=False)]
issues["byte-identical duplicate files"] = len(dup)
leak = dup.groupby("md5").split.nunique()
issues["duplicate groups spanning >1 split (leakage)"] = int((leak > 1).sum())
pd.Series(issues, name="count").to_frame()

In [ ]:
dup_view = dup.sort_values("md5")[["md5", "image", "split"]].assign(md5=lambda d: d.md5.str[:8])
dup_view.head(20)